# Qwen2.5-Coder C++ Review — QLoRA Training on Kaggle

Run top to bottom. **Cell 2 (`RUN CONTROL`) is the only cell you normally edit.**

**Fresh run** — `FRESH_START = True`. Anything already in `OUTPUT_DIR` is moved to
`outputs/.../archive/run-<timestamp>/` (moved, not deleted) and training begins at step 0.

**Continuing** — `FRESH_START = False`. Checkpoints are imported from `PREVIOUS_RUN_DIR`
and training picks up at the exact step it stopped at.

Either way, `training.resume_mode` stays `auto`: if this Kaggle session is killed
mid-run, re-running just the training cell continues where it left off. Freshness is a
property of the output directory, not a setting — so a crash can never be mistaken for
a request to start over.

In [ ]:
# ============================================================================
# RUN CONTROL — the only cell you normally edit
# ============================================================================
# True  -> train from step 0. Existing output is archived; no checkpoints imported.
# False -> continue a previous run by importing its checkpoints from PREVIOUS_RUN_DIR.
FRESH_START = True

# The task-tagged mixture from scripts/build_task_mixture.py, not the raw
# merged file. Update the dataset slug below to whatever you named the upload.
DATASET_FILE     = '/kaggle/input/datasets/saffiullah892/mydataset01/task_mixture.jsonl'
PROJECT_INPUT    = '/kaggle/input/datasets/saffiullah892/my-projectfiles01'
OUTPUT_DIR       = '/kaggle/working/outputs/qwen2.5-coder-1.5b-cpp-review-qlora'
PREVIOUS_RUN_DIR = '/kaggle/input/datasets/saffiullah892/qwen2-5-01/outputs/qwen2.5-coder-1.5b-cpp-review-qlora'

import shutil
import time
from pathlib import Path

output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)

if FRESH_START:
    existing = [path for path in output_dir.iterdir() if path.name != 'archive']
    if existing:
        archive = output_dir / 'archive' / time.strftime('run-%Y%m%d-%H%M%S')
        archive.mkdir(parents=True, exist_ok=True)
        for path in existing:
            shutil.move(str(path), str(archive / path.name))
        print(f'Archived {len(existing)} item(s) from the previous run to {archive}')
    print('FRESH START — training begins at step 0')
    print('After this run starts, set FRESH_START = False so a session restart resumes')
    print('instead of archiving your progress.')
else:
    source = Path(PREVIOUS_RUN_DIR)
    assert source.is_dir(), f'PREVIOUS_RUN_DIR not found: {source}'
    imported = 0
    for path in sorted(source.glob('checkpoint-*')):
        target = output_dir / path.name
        if not target.exists():
            shutil.copytree(path, target)
            imported += 1
    print(f'Imported {imported} checkpoint(s) from {source}')
    print('RESUME — training continues from the newest complete checkpoint')

present = sorted(path.name for path in output_dir.glob('checkpoint-*'))
print('Checkpoints in the output dir:', present or 'none')

In [ ]:
!nvidia-smi
!python --version
!python -m pip install --upgrade uv

## Copy Project to Writable Storage

Kaggle mounts `/kaggle/input` as read-only. Training needs to write configs, checkpoints, adapters, `.pth` files, and ONNX exports, so the project is copied to `/kaggle/working/project-files`.

In [ ]:
import shutil
from pathlib import Path

project_input = Path(PROJECT_INPUT)
work_repo = Path('/kaggle/working/project-files')

assert project_input.is_dir(), f'Project folder not found: {project_input}'
assert Path(DATASET_FILE).is_file(), f'Training dataset not found: {DATASET_FILE}'

# The dataset must hold the project tree itself. Kaggle expands an archive only
# when it is uploaded as one to extract; a .zip added as a plain file arrives as
# a .zip, and then nothing below is the code this notebook needs.
marker = project_input / 'src' / 'qwen_cpp_review' / 'prompt.py'
if not marker.exists():
    print(f'{project_input}\ndoes not contain src/qwen_cpp_review/prompt.py. It contains:\n')
    for item in sorted(project_input.iterdir())[:25]:
        print('   ', item.name + ('/' if item.is_dir() else ''))
    raise SystemExit(
        'Upload the CONTENTS of dist/kaggle-project-files.zip (let Kaggle extract it), '
        'not the zip as a file.'
    )

# /kaggle/working survives between sessions. A previous run's copy sits here and
# shadows the new upload, so replace it rather than merging into it - that is
# what makes a correct re-upload look like it changed nothing.
%cd /kaggle/working
if work_repo.exists():
    shutil.rmtree(work_repo)
    print('Removed the previous working copy')
shutil.copytree(project_input, work_repo)
%cd /kaggle/working/project-files

assert Path('pyproject.toml').exists(), 'pyproject.toml missing after copy'
print('Project copied to:', work_repo)
print('Training dataset:', DATASET_FILE)

## Verify the Project Files Support Resume

Kaggle serves project files from a dataset snapshot, which can lag behind the repository.
This fails fast rather than training for hours with the old code.

In [ ]:
import time
from pathlib import Path

# Every marker below must be present in the project files this notebook copied
# into /kaggle/working. If one is missing the upload is stale, and the failure
# it prevents is silent, so this cell stops the run instead of warning.
REQUIRED = [
    ('src/qwen_cpp_review/resume.py', None, 'the resume rewrite'),
    ('src/qwen_cpp_review/config.py', 'resume_mode', 'the resume config fields'),
    ('src/qwen_cpp_review/trainer.py', 'resolve_resume_plan', 'the resume rewrite'),
    ('src/qwen_cpp_review/prompt.py', 'TASKS = {', 'the task registry'),
    ('src/qwen_cpp_review/prompt.py', 'def has_field', 'the None-valued field guard'),
    ('src/qwen_cpp_review/prompt.py', 'def resolve_output_fields', 'per-task field resolution'),
    ('src/qwen_cpp_review/identifier_augmentation.py', 'apply_mapping_to_row', 'the anchor-safe rename'),
    ('src/qwen_cpp_review/prompt.py', 'def build_prompt_completion', 'the prompt/completion split'),
    ('src/qwen_cpp_review/trainer.py', 'def check_supervision_setup', 'the supervision guard'),
    ('scripts/verify_loss_masking.py', 'LOSS MASKING VERIFIED', 'the masking verification'),
]

stale = []
for relative, marker, what in REQUIRED:
    path = Path(relative)
    if not path.exists():
        stale.append((relative, f'file missing ({what})'))
    elif marker and marker not in path.read_text():
        stale.append((relative, f'missing {what}'))

if stale:
    print('STALE PROJECT FILES\n')
    for relative, why in stale:
        path = Path(relative)
        when = time.strftime('%Y-%m-%d %H:%M', time.localtime(path.stat().st_mtime)) if path.exists() else '-'
        print(f'  {relative:<48} {why}   (mtime {when})')

    print(f'\nWorking copy : {Path.cwd()}')
    print(f'Mounted from : {PROJECT_INPUT}')
    print('\nFix, in order:')
    print('  1. Upload dist/kaggle-project-files.zip as a NEW VERSION of the')
    print('     project-files dataset (not a new dataset, unless you also change')
    print('     PROJECT_INPUT above).')
    print('  2. In this notebook, open the right-hand Input panel and confirm the')
    print('     dataset shows the new version. Kaggle pins the version that was')
    print('     attached when the session started.')
    print('  3. Restart the session (Run -> Restart & clear cell outputs), then run')
    print('     from the top. /kaggle/input is mounted at session start, so a new')
    print('     dataset version is invisible to a session that is already running.')
    raise SystemExit('Stale project files - see the steps above')

print('Resume-capable, task-aware project files detected')
print('Working copy:', Path.cwd())

## Install Dependencies with uv

In [ ]:
!mkdir -p /kaggle/temp/uv-cache /kaggle/temp/project-venv /kaggle/temp/hf-cache /kaggle/temp/hf-datasets
!UV_PROJECT_ENVIRONMENT=/kaggle/temp/project-venv UV_CACHE_DIR=/kaggle/temp/uv-cache UV_LINK_MODE=copy uv sync --extra gpu --extra export --extra dev
!du -sh /kaggle/temp/project-venv /kaggle/temp/uv-cache /kaggle/working/project-files || true

## Configure Training

Writes the Kaggle paths into `configs/train_qlora.yaml`. `resume_mode` is left on `auto`
so an interrupted session resumes exactly; `optim` is pinned because a checkpoint's
optimizer state can only be reloaded by the optimizer that wrote it.

In [ ]:
import yaml
from pathlib import Path

config_path = Path('configs/train_qlora.yaml')
config = yaml.safe_load(config_path.read_text())

data = config['data']
data['data_files'] = [DATASET_FILE]
data['cache_dir'] = '/kaggle/temp/hf-datasets'
# Held for Phase 3, where variable-name robustness is the thing being measured.
# Enabling it here doubles every epoch for a result this run does not report.
data['identifier_augmentation'] = False
data['identifier_augmentation_copies'] = 1
# 66k rows: 5% is 3,305 eval rows, and every eval pass walks all of them. 1% is
# still ~660 rows, which is plenty to draw a loss curve.
data['validation_split_ratio'] = 0.01
# vram-profiles kaggle-t4. Costs 1.2% of rows, which the mixture builder already
# dropped, and roughly doubles throughput against 2048.
data['max_seq_length'] = 1024
# false = completion-only loss: only the target is supervised. It was true
# for the first run and nothing read it, so roughly half the gradient signal
# went into reproducing the instruction and the input code.
data['train_on_inputs'] = False

training = config['training']
training['output_dir'] = OUTPUT_DIR
# auto = continue from the newest usable checkpoint, start at 0 when there is none.
training['resume_mode'] = 'auto'
training['resume_from_checkpoint'] = None
training['initial_adapter_path'] = None
training['resume_auto_fallback'] = True
# Keep this fixed for the whole run — changing it makes saved optimizer state unloadable.
training['optim'] = 'adamw_torch'
# One epoch over 66k task-tagged rows. Three epochs on this much data overfits a
# LoRA long before it finishes, and will not fit a Kaggle session.
training['num_train_epochs'] = 1
# Effective batch 32 = 2 x 16, the vram-profiles kaggle-t4 target.
training['per_device_train_batch_size'] = 2
training['gradient_accumulation_steps'] = 16
training['logging_steps'] = 10
# At 50 the eval passes cost more than the training does. save_steps must stay a
# multiple of eval_steps for load_best_model_at_end.
training['save_steps'] = 250
training['eval_steps'] = 250
training['save_total_limit'] = 3
training['packing'] = False
# The built-in bar redraws with carriage returns; this cell is piped through
# tee, which records every redraw as a line. ThroughputAndMemoryCallback
# prints one complete progress line per logging step instead.
training['disable_tqdm'] = True
# Keep this on. Run 1 peaked at 6.36 GB of the T4's 15.8 GB *because* it was
# enabled; with it off, batch 2 / seq 1024 OOMs in the cross-entropy logits
# (2 x 1024 x 151936). The headroom was the effect, not spare capacity.
training['gradient_checkpointing'] = True
# Batch similar lengths together. A random batch pads a 29-token complexity
# target up to a ~700-token line_comments one: measured 35.2% of compute
# lost to padding, which grouping removes almost entirely.
training['group_by_length'] = True
training['gradient_checkpointing_use_reentrant'] = False
training['ddp_find_unused_parameters'] = False

config_path.write_text(yaml.safe_dump(config, sort_keys=False))

# --- run budget ----------------------------------------------------------- #
# A Kaggle session is 9-12 h and the weekly budget is ~30 GPU-h, so the cost of
# a run is worth knowing before it starts rather than after it is killed.
rows = sum(1 for line in open(DATASET_FILE) if line.strip())
val = int(rows * data['validation_split_ratio'])
train_rows = rows - val
effective_batch = training['per_device_train_batch_size'] * training['gradient_accumulation_steps']
samples = train_rows * training['num_train_epochs']
steps = samples // effective_batch
evals = max(1, steps // training['eval_steps'])

print('Dataset      :', data['data_files'])
print('Output dir   :', training['output_dir'])
print('Resume mode  :', training['resume_mode'])
print('Optimizer    :', training['optim'])
print()
print(f'Rows         : {rows:,}  ({train_rows:,} train / {val:,} eval)')
print(f'Epochs       : {training["num_train_epochs"]}')
print(f'Seq length   : {data["max_seq_length"]}')
print(f'Batch        : {training["per_device_train_batch_size"]} x '
      f'{training["gradient_accumulation_steps"]} = {effective_batch} effective')
print(f'Steps        : {steps:,}')
print(f'Eval passes  : {evals} x {val:,} rows')
print()
print('Rough wall-clock (the real rate is printed once training starts):')
for rate in (3, 5, 8):
    hours = samples / rate / 3600 + evals * val / (rate * 3) / 3600
    print(f'  at {rate} samples/s : {hours:.1f} h')
print('\nKaggle session limit is 9-12 h. If the estimate exceeds that, lower')
print('num_train_epochs or raise eval_steps before starting.')

## Optional Sanity Check

This checks that the dataset file is readable and shows the first row keys.

In [ ]:
import json
from collections import Counter
from pathlib import Path

dataset_path = Path(DATASET_FILE)
tasks = Counter()
rows = 0
with dataset_path.open() as handle:
    for line in handle:
        if not line.strip():
            continue
        rows += 1
        tasks[json.loads(line).get('task', '<none>')] += 1

print('Dataset size GB:', round(dataset_path.stat().st_size / 1024**3, 3))
print('Rows:', rows)
print('Tasks:')
for task, count in tasks.most_common():
    print(f'  {task:<16} {count}')

# A mixture with no `task` keys means the old merged file was uploaded, which
# trains every row on the full field list instead of one task at a time.
assert '<none>' not in tasks, 'This file has no task tags — upload cleaned/task_mixture.jsonl'

## Verify Loss Masking — do not skip

Wrong label masking is the most expensive silent failure in supervised fine-tuning:
the loss curve falls, checkpoints save, and the model learns to echo the instruction
back instead of answering it. Nothing raises.

The first real run trained with `train_on_inputs: true`, which supervised the whole
sequence — instruction and input C++ included. That is why it reported `eval_loss 0.46`
and perplexity 1.59 while mostly being scored on reproducing a prompt that is nearly
identical on every row.

This decodes a real batch through the real collator and asserts the supervised span is
the target only. It downloads no weights and takes well under a minute. A non-zero exit
means **do not train**.

In [ ]:
%%bash
set -euo pipefail
export UV_PROJECT_ENVIRONMENT=/kaggle/temp/project-venv
export UV_CACHE_DIR=/kaggle/temp/uv-cache
export HF_HOME=/kaggle/temp/hf-cache
export HF_DATASETS_CACHE=/kaggle/temp/hf-datasets
export NO_COLOR=1
uv run python scripts/verify_loss_masking.py \
  --config configs/train_qlora.yaml --rows 8 --show 1

## Resume Status

Read-only preview of exactly what the training cell will do: which checkpoint it picks,
which step it starts from, and whether the optimizer and LR schedule are restored.
Run it again any time — it never modifies anything.

In [ ]:
%%bash
export UV_PROJECT_ENVIRONMENT=/kaggle/temp/project-venv
export UV_CACHE_DIR=/kaggle/temp/uv-cache
export NO_COLOR=1
uv run python - <<'PY'
import sys
sys.argv = ['resume-status', '--config', 'configs/train_qlora.yaml']
from qwen_cpp_review.cli import resume_status_main
resume_status_main()
PY

## Start or Resume Training

Detects the GPU count and launches single- or multi-GPU Accelerate. The trainer prints a
`RESUME MODE:` banner stating the starting step and whether the optimizer state, LR
schedule and step counter were restored — read that banner to confirm what happened.

If this session is killed, just re-run this cell (with `FRESH_START = False`).

In [ ]:
%%bash
set -euo pipefail
export PYTHONUNBUFFERED=1
export HF_HOME=/kaggle/temp/hf-cache
export HF_DATASETS_CACHE=/kaggle/temp/hf-datasets
export UV_CACHE_DIR=/kaggle/temp/uv-cache
export UV_PROJECT_ENVIRONMENT=/kaggle/temp/project-venv
export ACCELERATE_LOG_LEVEL=info
export TRANSFORMERS_VERBOSITY=info
export NO_COLOR=1
# The OOM message recommends this; it reduces allocator fragmentation.
export PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True
LOG_FILE=/kaggle/working/train.log
echo "Training log: ${LOG_FILE}"
echo "Started at: $(date)" | tee -a "${LOG_FILE}"

NUM_GPUS=$(uv run python - <<'PY'
import torch
print(torch.cuda.device_count())
PY
)
echo "Detected GPUs: ${NUM_GPUS}" | tee -a "${LOG_FILE}"

if [ "${NUM_GPUS}" -gt 1 ]; then
  ACCEL_CONFIG=configs/accelerate_multi_gpu.yaml
  EXTRA_ARGS="--num_processes ${NUM_GPUS}"
else
  ACCEL_CONFIG=configs/accelerate_single_gpu.yaml
  EXTRA_ARGS=""
fi

uv run accelerate launch --config_file "${ACCEL_CONFIG}" ${EXTRA_ARGS} \
  train.py --config configs/train_qlora.yaml 2>&1 | tee -a "${LOG_FILE}"

## View Training Log

Run this after training finishes, or from the Kaggle console while training is running.

In [ ]:
# Run this any time - during training from a second browser tab, or after it
# ends. It reads the log file, so it never interferes with the running cell.
import re
from pathlib import Path

log = Path('/kaggle/working/train.log')
if not log.exists():
    print('No log yet:', log)
else:
    text = log.read_text(errors='replace')
    steps = re.findall(
        r'step (\d+)/(\d+) ep ([\d.]+) loss ([\d.]+).*?([\d.]+) it/s(?: ([\d,]+) tok/s)?'
        r'(?: \| mem ([\d.]+)/([\d.]+) GB)?.*?eta ([\d:\-]+)',
        text,
    )
    evals = re.findall(r'EVAL step (\d+)\s+eval_loss ([\d.]+)\s+perplexity ([\d.]+)', text)

    if steps:
        step, total, epoch, loss, its, toks, alloc, reserved, eta = steps[-1]
        first_loss = steps[0][3]
        print(f'PROGRESS   step {step}/{total}   epoch {epoch}   {100 * int(step) / int(total):.1f}%')
        print(f'LOSS       {first_loss} -> {loss}   ({len(steps)} log points)')
        print(f'SPEED      {its} it/s' + (f'   {toks} tok/s' if toks else ''))
        if alloc:
            print(f'GPU MEM    {alloc} GB allocated / {reserved} GB reserved  (T4 has 15.8 GB usable)')
        print(f'ETA        {eta}')
    else:
        print('No progress lines yet - training may still be loading the model.')

    if evals:
        print('\nEVAL HISTORY')
        for s, el, pp in evals[-8:]:
            print(f'  step {s:>6}   eval_loss {el}   perplexity {pp}')
        losses = [float(e[1]) for e in evals]
        if len(losses) > 1:
            trend = 'improving' if losses[-1] < min(losses[:-1]) else 'NOT improving'
            print(f'  latest is {trend} vs best previous {min(losses[:-1]):.4f}')

    print('\n--- last 25 lines ---')
    for line in text.splitlines()[-25:]:
        print(line)

## Check Saved Training Outputs

Expected outputs include `best_adapter/`, `best_adapter.pth`, `last_adapter/`, `last_adapter.pth`, `final_adapter/`, and `final_adapter.pth`.

In [ ]:
!ls -lah /kaggle/working/outputs/qwen2.5-coder-1.5b-cpp-review-qlora || true
!find /kaggle/working/outputs/qwen2.5-coder-1.5b-cpp-review-qlora -maxdepth 2 -type f \( -name '*.pth' -o -name 'adapter_model.safetensors' -o -name 'trainer_state.json' \) -print || true

## Evaluate Best Adapter

In [ ]:
!UV_PROJECT_ENVIRONMENT=/kaggle/temp/project-venv UV_CACHE_DIR=/kaggle/temp/uv-cache uv run python evaluate.py \
  --config configs/train_qlora.yaml \
  --adapter /kaggle/working/outputs/qwen2.5-coder-1.5b-cpp-review-qlora/best_adapter

## Smoke-Test the Trained Adapter

Runs easy/medium/hard C++ examples plus renamed-variable checks through the adapter that
`scripts/test_model.py` points at. Update the checkpoint path in that script to match the
adapter you want to test (`best_adapter`, `last_adapter`, or a specific `checkpoint-N`).

In [ ]:
!UV_PROJECT_ENVIRONMENT=/kaggle/temp/project-venv UV_CACHE_DIR=/kaggle/temp/uv-cache uv run python scripts/test_model.py
!head -2 /kaggle/working/outputs/model_test_predictions.jsonl || true

## Merge Best LoRA Adapter

In [ ]:
!UV_PROJECT_ENVIRONMENT=/kaggle/temp/project-venv UV_CACHE_DIR=/kaggle/temp/uv-cache uv run python merge_lora.py \
  --base-model Qwen/Qwen2.5-Coder-1.5B-Instruct \
  --adapter /kaggle/working/outputs/qwen2.5-coder-1.5b-cpp-review-qlora/best_adapter \
  --output-dir /kaggle/working/outputs/qwen2.5-coder-1.5b-cpp-review-merged

## Export Merged Model to ONNX

In [ ]:
!UV_PROJECT_ENVIRONMENT=/kaggle/temp/project-venv UV_CACHE_DIR=/kaggle/temp/uv-cache uv run python export_onnx.py \
  --model /kaggle/working/outputs/qwen2.5-coder-1.5b-cpp-review-merged \
  --output /kaggle/working/outputs/qwen2.5-coder-1.5b-cpp-review.onnx

## Final Files to Download

Download from `/kaggle/working/outputs` after training finishes.

In [ ]:
!find /kaggle/working/outputs -maxdepth 3 -type f \( -name '*.pth' -o -name '*.onnx' -o -name 'adapter_model.safetensors' -o -name 'model.safetensors' -o -name 'training_config.yaml' \) -print || true